# Qwen-Image-2.1 via stable-diffusion.cpp — Colab T4

Run **Qwen-Image-2.1** on free Colab (T4) with
[stable-diffusion.cpp](https://github.com/leejet/stable-diffusion.cpp)
(day-0 Qwen-Image-2.1 support since 2026-09-20).

**Models used** (all official releases):
| role | file | size |
|---|---|---|
| diffusion (UNet) | `qwen_image_2.1_int8_convrot.safetensors` (INT8 convrot, T4-native, fastest) **or** `qwen_image_2.1-Q5_0.gguf` (GGUF, memory-efficient, slower) — select in cell ④ | 7.26 / 5.07 GB |
| text encoder | `Qwen3VL-8B-Instruct-Q4_K_M.gguf` | 5.03 GB |
| vision (Edit mode only) | `mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf` → `--llm_vision` | 0.75 GB |
| VAE | `vae/qwen_image_2.1_vae_bf16.safetensors` | 0.68 GB |
| turbo LoRA (optional) | `Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors` | 0.68 GB |

**Turbo mode** (checkbox in cell ⑥, enabled by default): Viggle v0.3 r128 LoRA,
applied unmerged at runtime by sd.cpp (as Viggle recommends — do not merge it).
Recipe: **6 steps**, **cfg 1.0**, no negative prompt, LoRA strength 1.0. The sigma
schedule replicates `ViggleTurboSigmas` from Viggle's official repository: six nodes
`[1.0, 0.9375, 0.875, 0.75, 0.5, 0.25]` with resolution-dependent dynamic shift,
plus an appended terminal 0 → 7 sigmas = 6 genuine steps (computed automatically
in cell ⑥ from the canvas size). Unchecking `turbo_sigmas` falls back to sd.cpp's
default schedule (for experiments). The base UNet is the plain upstream base
(not turbo-merged) — the LoRA is what makes it fast.

**Notes:**
- There is no official Linux CUDA binary (Windows CUDA only) → cell ③ downloads
  a prebuilt binary (built by our own GitHub Actions, CUDA 12.6, ~718 MB,
  SHA-256 verified). Once per runtime (approx. 2–5 min).
- T4 = compute capability 7.5 → the native INT8 convrot CUDA path is used.
  No Python/torch at all — pure `sd-cli`.
- Total weights **≈14.4 GB < 15 GB VRAM** (INT8 variant) → fits fully in VRAM;
  the `offload_to_cpu` checkbox exists but is **off by default** — verified on
  T4: enabling it gets the process killed by the OOM killer (Colab RAM is only
  12.7 GB). Enable only if your VRAM is smaller than a T4's.
- **UNet variants** (dropdown in cell ④): INT8-convrot is fastest on T4;
  GGUF Q5_0 saves ≈2 GB of VRAM but is slower. The INT8 + turbo LoRA combination
  is verified working; the GGUF variant has not been GPU-tested yet.
- **Edit mode requires `unet_offload_ram`** on T4: UNet weights are staged
  layer-wise from system RAM on demand, while TE/VAE stay on GPU for fast
  encoding. Without it, VRAM spikes. Do not enable together with
  `te_offload_cpu`.
- **Canvas**: preset dropdown in cell ⑥ (8 presets ≈1MP, multiples of 32, plus
  Custom for manual size). In Edit mode, `auto_size_ref1` follows the reference
  aspect ratio.
- The GGUF text encoder is text-only → **Edit** mode requires `--llm_vision`
  (mmproj, automatic in cell ⑥); T2I mode does not need it.
- Image dimensions **must be multiples of 32** (handled automatically in cell ⑥).

**Cell order:** ① GPU check → ② install downloader → ③ download prebuilt sd-cli →
④ download models → ⑤ upload references (optional) → ⑥ generate → ⑦ batch (optional).

> Cells ③ & ④ are guarded: re-running will not re-download.

> **Note:** the recipe (flags, 6-step turbo, VRAM fit) is verified working on a
> free Colab T4 (txt2img and image editing) with the official model weights
> above, using the default INT8-convrot configuration. The GGUF Q5_0 UNet
> alternative has not been GPU-tested yet.



In [ ]:
#@title ① Check GPU / disk { vertical-output: true }
import shutil, subprocess
print("=== nvidia-smi ===")
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout.strip() or "nvidia-smi not found — select Runtime > Change runtime type > GPU!")
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"disk /content free: {free_gb:.1f} GB")
need = 20.0
print(("OK, enough for models (14.4 GB) + sd-cli." if free_gb >= need
       else f"WARNING: need ~{need:.0f} GB free, only {free_gb:.1f} GB available!"))



In [ ]:
#@title ② Install downloader (once per runtime) { vertical-output: true }
# No torch/transformers or compiler needed — sd-cli ships as a prebuilt
# binary; only a downloader is required (aria2 for the models).
!apt-get update -qq && apt-get install -y -qq aria2 > /dev/null 2>&1
!aria2c --version | head -1



In [ ]:
#@title ③ Download sd-cli prebuilt (CUDA 12.6) { vertical-output: true }
#@markdown Once per runtime (approx. 2–5 min). Binary built by GitHub Actions
#@markdown (sd.cpp sdcpp-master-929-3f8527a-cuda12.6, CUDA 12.6, sm_75;80;86;89;90,
#@markdown CUDA runtime bundled in lib/). Skipped automatically when already present.
%%bash
set -e
URL="https://github.com/fendimaharsi0-sketch/sdcpp-cuda-linux/releases/download/sdcpp-master-929-3f8527a-cuda12.6/sdcpp-sd-cli-cuda12.6-ubuntu22.04-x86_64.tar.gz"
SHA="1b19a8815c2f06ed36f963648d621ea593a54f2778df09dbb21e3ead781b45eb"
DEST="/content/sdcpp-cuda"
SH="$DEST/sd-cli.sh"
if [ -x "$SH" ]; then
  echo "sd-cli already present — skipping download."
  "$SH" --help 2>&1 | head -3
  exit 0
fi
mkdir -p "$DEST"
cd "$DEST"
echo "== download prebuilt sd-cli (~718 MB) =="
wget -q --show-progress -O sdcpp.tar.gz "$URL"
echo "== verify sha256 =="
echo "$SHA  sdcpp.tar.gz" | sha256sum -c -
echo "== extract =="
tar xzf sdcpp.tar.gz --no-same-owner
rm sdcpp.tar.gz
chmod +x "$DEST/sd-cli.sh" "$DEST/bin/sd-cli"
echo "== done =="
ls -la "$DEST/bin/sd-cli" "$DEST/lib"
"$SH" --help 2>&1 | head -3



In [ ]:
#@title ④ Download models (auto-skip if present) { vertical-output: true }
import os, subprocess
unet_variant = "INT8-convrot (fastest)" #@param ["INT8-convrot (fastest)", "GGUF Q5_0 (memory-efficient)"]
# (repo, file name in repo, exact byte size from the HF API, local folder)
UNET_FILES = {
    "INT8-convrot (fastest)":
        ("Comfy-Org/Qwen-Image-2.1",
         "diffusion_models/qwen_image_2.1_int8_convrot.safetensors", 7256783064, "/content/models/diffusion"),
    "GGUF Q5_0 (memory-efficient)":
        ("leejet/Qwen-Image-2.1-GGUF",
         "qwen_image_2.1-Q5_0.gguf", 5069910048, "/content/models/diffusion"),
}
FILES = [
    ("Qwen/Qwen3-VL-8B-Instruct-GGUF",
     "Qwen3VL-8B-Instruct-Q4_K_M.gguf", 5027784800, "/content/models/te"),
    ("Qwen/Qwen3-VL-8B-Instruct-GGUF",
     "mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf", 752289728, "/content/models/te"),
    ("Comfy-Org/Qwen-Image-2.1",
     "vae/qwen_image_2.1_vae_bf16.safetensors", 675509688, "/content/models/vae"),
    ("Viggle/Qwen-Image-2.1-viggle-turbo",
     "Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors", 679604800, "/content/models/lora"),
]
# the selected UNet is inserted first so it downloads first (largest)
FILES.insert(0, UNET_FILES[unet_variant])
for repo, fname, size, d in FILES:
    os.makedirs(d, exist_ok=True)
    base = os.path.basename(fname)
    p = os.path.join(d, base)
    if os.path.exists(p) and os.path.getsize(p) == size:
        print(f"OK skip {base} ({size/1e9:.2f} GB already present)")
        continue
    url = f"https://huggingface.co/{repo}/resolve/main/{fname}"
    print(f"download {base} ({size/1e9:.2f} GB) ...")
    subprocess.run(["aria2c", "-x", "16", "-s", "16", "-c", "--console-log-level=warn",
                    "--summary-interval=30", "-d", d, "-o", base, url], check=True)
    got = os.path.getsize(p)
    assert got == size, f"SIZE MISMATCH {base}: {got} != {size}; delete the file and re-run this cell"
    print(f"OK {base}")
# record the UNet choice for cell ⑥ to read
open("/content/models/diffusion/.unet_choice", "w").write(os.path.basename(UNET_FILES[unet_variant][1]))
print(f"\nselected unet: {os.path.basename(UNET_FILES[unet_variant][1])}")
print("All models ready.")



In [ ]:
#@title ⑤ Upload reference images (optional — for Edit mode) { vertical-output: true }
#@markdown Upload 1–2 images, then write their **file names** into the `ref1`/`ref2` params
#@markdown of cell ⑥. Formats: jpg/png/webp/bmp.
import os
os.makedirs("/content/refs", exist_ok=True)
from google.colab import files
up = files.upload()
for name, data in up.items():
    open(f"/content/refs/{name}", "wb").write(data)
    print("saved:", name)
print()
!ls -la /content/refs



In [ ]:
#@title ⑥ Generate { vertical-output: true }
#@markdown ### Mode & prompt
mode = "Text-to-Image" #@param ["Text-to-Image", "Edit (1-2 ref)"]
prompt = "a lovely cat holding a sign that says 'qwen2.1.cpp'" #@param {type:"string"}
negative_prompt = "" #@param {type:"string"}
#@markdown ### References (Edit mode only — file names under /content/refs)
ref1 = "" #@param {type:"string"}
ref2 = "" #@param {type:"string"}
#@markdown ### Turbo (LoRA Viggle v0.3, 6 step)
turbo = True #@param {type:"boolean"}
#@markdown ### Custom Viggle sigmas (turbo only; off = sd.cpp default schedule)
turbo_sigmas = True #@param {type:"boolean"}
#@markdown ### Size & sampling (ignored when turbo is on)
canvas_preset = "1:1 1024x1024" #@param ["1:1 1024x1024", "16:9 1344x768", "9:16 768x1344", "3:2 1248x832", "2:3 832x1248", "4:3 1184x896", "3:4 896x1184", "21:9 1568x672", "Custom (manual width/height)"]
auto_size_ref1 = True #@param {type:"boolean"}
width = 1024 #@param {type:"integer"}
height = 1024 #@param {type:"integer"}
steps = 20 #@param {type:"integer"}
cfg_scale = 1.0 #@param {type:"number"}
sampler = "euler" #@param ["euler", "euler_a", "heun", "dpm2", "dpmpp_2m", "dpmpp_2s_a", "lcm"]
seed = -1 #@param {type:"integer"}
#@markdown ### Memory
offload_to_cpu = False #@param {type:"boolean"}
prefix_cache = "auto" #@param ["auto", "q8_0", "off"]
te_offload_cpu = False #@param {type:"boolean"}
unet_offload_ram = False #@param {type:"boolean"}

import os, subprocess, random
from PIL import Image as PILImage
from IPython.display import Image, display

SD = "/content/sdcpp-cuda/sd-cli.sh"  # wrapper: sets LD_LIBRARY_PATH to the bundled lib/
# DIFF follows the UNet choice from cell ④ (written to the marker file at download time)
_marker = "/content/models/diffusion/.unet_choice"
assert os.path.exists(_marker), "UNet choice not found — run cell ④ first!"
with open(_marker) as _f:
    DIFF = "/content/models/diffusion/" + _f.read().strip()
print("diffusion model:", DIFF)
LLM  = "/content/models/te/Qwen3VL-8B-Instruct-Q4_K_M.gguf"
MMPROJ = "/content/models/te/mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf"
VAE  = "/content/models/vae/qwen_image_2.1_vae_bf16.safetensors"
LORA = "/content/models/lora/Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors"
for p, label in [(SD, "sd-cli"), (DIFF, "diffusion"), (LLM, "text encoder"), (VAE, "vae")]:
    assert os.path.exists(p), f"{label} not found at {p} — run cell ③/④ first!"

# --- turbo mode: official Viggle v0.3 recipe (unmerged LoRA, 6 steps, cfg 1.0) ---
# raw nodes (unshifted); final schedule = ViggleTurboSigmas-style dynamic shift + appended 0
TURBO_NODES = [1.0, 0.9375, 0.875, 0.75, 0.5, 0.25]
def viggle_sigmas(w, h):
    import math
    tokens = (w // 16) * (h // 16)
    mu = 0.5 + (0.9 - 0.5) * (tokens - 256) / (8192 - 256)
    em = math.exp(mu)
    shifted = [em / (em + (1.0 / t - 1.0)) for t in TURBO_NODES] + [0.0]
    return ",".join(f"{s:.4f}" for s in shifted)
if turbo:
    assert os.path.exists(LORA), f"turbo LoRA not found at {LORA} — run cell ④ first!"
    prompt = prompt.strip() + f" <lora:{os.path.basename(LORA)}:1.0>"
    steps, cfg_scale, sampler = 6, 1.0, "euler"
    if negative_prompt.strip():
        print("note: turbo ignores the negative prompt.")
        negative_prompt = ""
    print(f"TURBO ON: 6 steps, sigmas {'custom Viggle' if turbo_sigmas else 'sd.cpp default'}, cfg 1.0, LoRA r128 @1.0 (runtime)")

def snap32(x):
    return max(32, (int(x) // 32) * 32)

def guard_ref(p):
    # limit reference input to ~1MP (LANCZOS, keep aspect) to bound KV/activation memory
    im = PILImage.open(p)
    w, h = im.size
    if w * h > 1024 * 1024:
        s = (1024 * 1024 / (w * h)) ** 0.5
        nw, nh = int(w * s), int(h * s)
        im = im.resize((nw, nh), PILImage.LANCZOS)
        os.makedirs("/content/refs_guarded", exist_ok=True)
        g = os.path.join("/content/refs_guarded", os.path.basename(p))
        im.save(g)
        print(f"guard input: {os.path.basename(p)} {w}x{h} -> {nw}x{nh}")
        return g
    return p

refs = []
# --- canvas preset: convenience dropdown; "Custom" = use manual width/height ---
# (auto_size_ref1 in Edit mode still takes precedence when enabled)
CANVAS_PRESETS = {
    "1:1 1024x1024": (1024, 1024),
    "16:9 1344x768": (1344, 768),
    "9:16 768x1344": (768, 1344),
    "3:2 1248x832": (1248, 832),
    "2:3 832x1248": (832, 1248),
    "4:3 1184x896": (1184, 896),
    "3:4 896x1184": (896, 1184),
    "21:9 1568x672": (1568, 672),
}
if canvas_preset in CANVAS_PRESETS:
    width, height = CANVAS_PRESETS[canvas_preset]
    print(f"canvas preset: {canvas_preset} -> {width}x{height}")
if mode.startswith("Edit"):
    for r in (ref1.strip(), ref2.strip()):
        if r:
            p = os.path.join("/content/refs", r)
            assert os.path.exists(p), f"ref not found: {p} (upload it in cell ⑤ first)"
            refs.append(guard_ref(p))
    assert refs, "Edit mode needs at least 1 ref — fill in ref1!"
    if auto_size_ref1:
        # canvas follows the ref1 aspect ratio, ~1MP, multiples of 32
        w0, h0 = PILImage.open(refs[0]).size
        s = (1024 * 1024 / (w0 * h0)) ** 0.5
        width, height = snap32(w0 * s), snap32(h0 * s)
        print(f"auto size from ref1: {w0}x{h0} -> {width}x{height}")
    # GGUF TE is text-only -> Edit mode needs the vision tower via --llm_vision
    assert os.path.exists(MMPROJ), f"mmproj not found at {MMPROJ} — run cell ④ first!"
else:
    width, height = snap32(width), snap32(height)

if seed < 0:
    seed = random.randint(0, 2**31 - 1)
print(f"seed: {seed}")

os.makedirs("/content/outputs", exist_ok=True)
# auto output name: unique per run (mode + timestamp) — no overwrite risk
from datetime import datetime
_tag = "edit" if mode.startswith("Edit") else "txt2img"
_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"/content/outputs/qwen21_{_tag}_{_ts}.png"
print("output:", out_path)

if unet_offload_ram and te_offload_cpu:
    raise ValueError("unet_offload_ram and te_offload_cpu cannot both be enabled — pick one!")

cmd = [SD,
       "--diffusion-model", DIFF,
       "--llm", LLM,
       "--vae", VAE,
       "--lora-model-dir", "/content/models/lora",  # read via <lora:name:1.0> in the prompt
       "-p", prompt,
       "-W", str(width), "-H", str(height),
       "--steps", str(steps),
       "--cfg-scale", str(cfg_scale),
       "--sampling-method", sampler,
       "-s", str(seed),
       "--fa",                # flash attention (CUDA)
       "-v"]
if turbo and turbo_sigmas:
    sig_arg = viggle_sigmas(width, height)  # computed from the final canvas (follows preset/auto_size)
    cmd += ["--sigmas", sig_arg]
    print(f"Viggle sigmas: {sig_arg} (7 sigmas = 6 steps, shift mu from {width}x{height})")
if te_offload_cpu:
    # TE (5.0 GB) to CPU: saves ~5.0 GB VRAM (12.7 GB RAM is safe),
    # useful for the more demanding Edit mode.
    cmd += ["--backend", "diffusion=cuda0,te=cpu",
            "--params-backend", "diffusion=cuda0,te=cpu"]
if unet_offload_ram:
    # UNet weights to system RAM, staged to GPU per block on demand
    # (layer-wise); TE/VAE stay on GPU for fast text/vision encoding.
    # Saves ~7 GB VRAM for Edit mode. Cost: PCIe transfer each step.
    # Required on T4 for Edit mode (VRAM spikes otherwise).
    cmd += ["--backend", "diffusion=cuda0,te=cuda0",
            "--params-backend", "diffusion=cpu"]
if offload_to_cpu:
    # CAUTION: verified on Colab T4 to get the process killed by the OOM killer
    # (12.7 GB RAM < 14.4 GB total weights). Enable only if VRAM < 14 GB.
    cmd += ["--offload-to-cpu"]
if negative_prompt.strip():
    cmd += ["-n", negative_prompt]
for r in refs:
    cmd += ["-r", r]
if mode.startswith("Edit"):
    cmd += ["--llm_vision", MMPROJ]  # vision tower, required for Edit
if prefix_cache == "q8_0":
    cmd += ["--model-args", "qwen_image_2_1_prefix_cache_type=q8_0"]
elif prefix_cache == "off":
    cmd += ["--model-args", "qwen_image_2_1_prefix_cache=false"]
cmd += ["-o", out_path]

print("\n$ " + " ".join(f"'{c}'" if " " in c else c for c in cmd) + "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
assert rc == 0 and os.path.exists(out_path), f"generate gagal (rc={rc})"
print("\n=== RESULT:", out_path, "===")
display(Image(out_path))



### ⑦ Batch (optional)

Repeat generation with different seeds without changing other params. Edit `n`, then run.



In [ ]:
#@title ⑦ Batch — N variations with different seeds { vertical-output: true }
#@markdown Re-runs the cell ⑥ command with different random seeds.
#@markdown Make sure cell ⑥ has succeeded at least once before using this.
n = 4 #@param {type:"integer"}

import subprocess, random, os
from IPython.display import Image, display

# reuse the last params from cell ⑥ (variables persist in the namespace)
batch_seeds = [random.randint(0, 2**31 - 1) for _ in range(n)]
base_cmd = cmd[:-2]  # cmd from cell ⑥, without "-o out_path"
base_out = out_path.rsplit(".", 1)
for i, sd in enumerate(batch_seeds):
    op = f"{base_out[0]}_batch{i+1}.{base_out[1]}"
    c = [a for a in base_cmd]
    si = c.index("-s")
    c[si + 1] = str(sd)
    c += ["-o", op]
    print(f"\n--- batch {i+1}/{n} seed={sd} ---")
    r = subprocess.run(c, capture_output=True, text=True)
    print(r.stdout[-1500:] if r.stdout else "")
    if r.returncode == 0 and os.path.exists(op):
        display(Image(op))
    else:
        print("FAILED:", r.stderr[-500:] if r.stderr else "?")
        break

